In [1]:
# ============================================================
# [CELL 00] — CONFIG & IMPORTS
# Projet POKEMON — Decks Compétitifs
# Prismora Solutions — v2.0 — 2026
# Notebook : 03_DECKS.ipynb
# ============================================================

import unicodedata
import requests
import pandas as pd
import json
import re
import time
import urllib.parse
from pathlib import Path
from bs4 import BeautifulSoup
from difflib import SequenceMatcher
from datetime import date

pd.set_option('future.no_silent_downcasting', True)

ROOT        = Path.cwd()
CONFIG_PATH = ROOT / 'config' / 'sources.json'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

CARDS_PATH = ROOT / CONFIG['output']['cards_pocket']
df_cards   = pd.read_csv(CARDS_PATH, encoding='utf-8-sig')

# Index nom_en normalisé → ligne(s) cards_pocket_fr
def normalise(s):
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z0-9]', '', s.lower())

cards_index_en = {}
for _, row in df_cards.iterrows():
    key = normalise(row['nom_en'])
    cards_index_en.setdefault(key, []).append(row)

BASE_META = 'https://www.pokemonmeta.com/api/v1'
HEADERS   = {'User-Agent': 'Mozilla/5.0', 'Referer': 'https://www.pokemonmeta.com/'}

OUT_DECKS_PTCG = ROOT / 'data/processed/decks_ptcgpocket.csv'
OUT_CARDS_PTCG = ROOT / 'data/processed/deck_cards_ptcgpocket.csv'
OUT_DECKS_META = ROOT / 'data/processed/decks_pokemonmeta.csv'
OUT_CARDS_META = ROOT / 'data/processed/deck_cards_pokemonmeta.csv'

# --- Fonctions utilitaires ---

def normalise_card_id(card_id_dotgg):
    m = re.match(r'^([A-Za-z0-9]+)-(\d+)$', card_id_dotgg)
    if not m:
        return None, None
    code_raw = m.group(1)
    numero   = int(m.group(2))
    code_set = 'PROMO-A' if code_raw == 'PROMO' else code_raw
    return code_set, numero

def extraire_slug(url):
    m = re.search(r'/decks/([^/]+)/?$', url)
    return m.group(1) if m else ''

def construire_nom_fr_deck(id_deck, df_cards_source):
    """
    Identifie les 1-2 Pokémon EX principaux du deck et construit un nom FR.
    Fonctionne quelle que soit la source (type_carte optionnel).
    """
    cartes_deck = df_cards_source[df_cards_source['id_deck'] == id_deck].copy()
    if cartes_deck.empty:
        return ''
    if 'type_carte' in cartes_deck.columns:
        is_poke = cartes_deck['type_carte'] == 'pokemon'
    else:
        is_poke = cartes_deck['id_carte'].str.contains('PKT-', na=False) & \
                  ~cartes_deck['nom_fr'].str.contains(
                      'Ball|Bonbon|Recherche|Copieuse|Hélio|Morgane|Casque|Cape|'
                      'Glace|Stade|Zone|Chemin|Mars|Atalante|Vitesse|Nettoyage|'
                      'Poncho|Pic|Barb|Candy|Trail|Dame|Sabrina|Cyrus|Lisia|'
                      'Korrina|Volkner|Brock|Professor|Poké|Rocky|Lucky|Hiking|'
                      'Training|Arena|Starting|Fragrant|Clear|Guzma|Colisée|'
                      'Plaine|Canot|Œuf|Cornélia',
                      case=False, na=False
                  )
    pokes = cartes_deck[is_poke].copy()
    if pokes.empty:
        return ''
    if 'quantite' in pokes.columns:
        pokes = pokes.sort_values('quantite', ascending=False)
    pokes_ex    = pokes[pokes['nom_fr'].str.contains('ex|EX', case=False, na=False)]
    pokes_nonex = pokes[~pokes['nom_fr'].str.contains('ex|EX', case=False, na=False)]
    noms = []
    for _, r in pokes_ex.head(2).iterrows():
        noms.append(r['nom_fr'])
    if len(noms) < 2 and len(pokes_nonex) > 0:
        noms.append(pokes_nonex.iloc[0]['nom_fr'])
    return ' / '.join(dict.fromkeys(noms)) if noms else ''

def nom_fr_depuis_nom_en(nom_en):
    """
    Construit un nom FR depuis un nom EN de deck.
    Ex: 'Mega Altaria ex Darkrai' → 'Méga-Altaria-ex / Darkrai'
    Matching trigramme/bigramme/unigramme sur cards_index_en.
    """
    mots_ignores = {'and', 'ex', 'mega', 'the', 'of'}
    mots    = nom_en.split()
    noms_fr = []
    i = 0
    while i < len(mots):
        trouve = False
        for taille in [3, 2, 1]:
            segment = ' '.join(mots[i:i+taille])
            key     = normalise(segment)
            if key in mots_ignores:
                i += 1
                trouve = True
                break
            matches = cards_index_en.get(key, [])
            if matches:
                nom_fr = matches[0]['nom_fr']
                if nom_fr and nom_fr not in noms_fr:
                    noms_fr.append(nom_fr)
                i += taille
                trouve = True
                break
        if not trouve:
            i += 1
    return ' / '.join(noms_fr[:2]) if noms_fr else nom_en

def export_to_sheets(df, key, gc, SHEETS_IDS):
    """Export DataFrame vers Google Sheets — robuste Int64 nullable."""
    sheet_id  = SHEETS_IDS[key]
    sh        = gc.open_by_key(sheet_id)
    worksheet = sh.sheet1
    df_clean  = df.copy()
    for col in df_clean.columns:
        if str(df_clean[col].dtype).startswith('Int') or pd.api.types.is_integer_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].astype(object)
    df_clean = df_clean.infer_objects(copy=False).fillna('').astype(str)
    data     = [df_clean.columns.tolist()] + df_clean.values.tolist()
    worksheet.clear()
    worksheet.update(data, value_input_option='RAW')
    print(f'  ✅ {key:25s} → {len(data)-1} lignes')

# Contrôle
print('=' * 55)
print('  PROJET POKEMON — Decks Compétitifs v2.0')
print('  Prismora Solutions — 03_DECKS.ipynb')
print('=' * 55)
print(f'\n📁 Racine          : {ROOT}')
print(f'✅ Cartes chargées  : {len(df_cards)}')
print(f'✅ Index EN         : {len(cards_index_en)} entrées')

C:\Users\JPVT\AppData\Local\Temp\ipykernel_10348\3532039243.py:20: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


  PROJET POKEMON — Decks Compétitifs v2.0
  Prismora Solutions — 03_DECKS.ipynb

📁 Racine          : d:\Github\pokemon-bdd
✅ Cartes chargées  : 3405
✅ Index EN         : 1253 entrées


In [2]:
# ============================================================
# [CELL 01] — SCRAPING TIER LIST ptcgpocket.gg
# ============================================================

def scrape_tier_list_ptcgpocket(url):
    resp = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    soup  = BeautifulSoup(resp.text, 'html.parser')
    decks = []
    table = soup.find('table')
    if table:
        for row in table.find_all('tr'):
            cols = row.find_all('td')
            if len(cols) < 2:
                continue
            tier_text = cols[0].get_text(strip=True)
            lien = cols[1].find('a')
            if not lien:
                continue
            nom  = lien.get_text(strip=True)
            href = lien.get('href', '')
            tier = re.sub(r'(.+?)\1+', r'\1', tier_text).strip()
            tier = re.sub(r'\s*Tier\s*', '', tier).strip()
            texte = cols[1].get_text()
            tendance = 'up' if '🔼' in texte else ('down' if '🔽' in texte else ('new' if '🆕' in texte else ''))
            decks.append({'source': 'ptcgpocket', 'tier': tier, 'nom': nom, 'url': href, 'tendance': tendance})
    return decks

decks_tl = scrape_tier_list_ptcgpocket('https://ptcgpocket.gg/tier-list/')

with open(ROOT / 'data/raw/tier_list_ptcgpocket.json', 'w', encoding='utf-8') as f:
    json.dump(decks_tl, f, ensure_ascii=False, indent=2)

print(f'✅ {len(decks_tl)} decks extraits\n')
print(pd.DataFrame(decks_tl)[['tier', 'nom', 'tendance']].to_string(index=False))

✅ 26 decks extraits

tier                             nom tendance
   S           Suicune ex Baxcalibur         
   S       Mega Altaria ex Gourgeist         
   S                 Mega Lucario ex         
   S          Mega Altaria ex Espeon         
   S        Mega Lucario ex Greninja         
   S       Mega Blaziken ex Greninja         
   S       Mega Manectrix ex Zeraora         
   S        Zoroark ex Mega Absol ex         
   S         Hydreigon Mega Absol ex         
   S       Mega Lucario ex Hitmontop         
   S           Miraidon ex Magnezone         
   S           Magnezone Miraidon ex         
   S          Greninja Mega Absol ex         
   S       Mega Sceptile ex Sceptile         
   S        Mega Altaria ex Greninja         
   A        Magnezone ex Miraidon ex         
   A       Iron Valiant Iron Boulder         
   A       Mega Charizard Y Entei ex         
   A       Mega Altaria ex Igglybuff         
   A        Mega Scizor ex Revavroom         
   A         

In [3]:
# ============================================================
# [CELL 02] — DECKLISTS ptcgpocket via API dotgg
# ============================================================

def get_decklist(slug):
    url  = f'https://api.dotgg.gg/cgfw/getdeck?game=pokepocket&slug={slug}'
    resp = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    return resp.json()

print('🔍 Récupération decklists via API dotgg...\n')
decks_avec_cartes = []

for deck in decks_tl:
    slug = extraire_slug(deck['url'])
    if not slug:
        print(f"  ⚠️  Slug vide : {deck['nom']}")
        continue
    try:
        data   = get_decklist(slug)
        cartes = []
        for card_id_raw, qty in data.get('deck', {}).items():
            code_set, numero = normalise_card_id(card_id_raw)
            if code_set is None:
                continue
            match = df_cards[(df_cards['code_set'] == code_set) & (df_cards['numero'] == numero)]
            if len(match) == 1:
                r = match.iloc[0]
                cartes.append({'card_id_dotgg': card_id_raw, 'code_set': code_set, 'numero': numero,
                               'id_carte': r['id_carte'], 'nom_fr': r['nom_fr'], 'nom_en': r['nom_en'],
                               'type_carte': r['type_carte'], 'quantite': int(qty)})
            else:
                cartes.append({'card_id_dotgg': card_id_raw, 'code_set': code_set, 'numero': numero,
                               'id_carte': 'NON_MATCHÉ', 'nom_fr': '', 'nom_en': '', 'type_carte': '', 'quantite': int(qty)})
        nb_ko  = sum(1 for c in cartes if c['id_carte'] == 'NON_MATCHÉ')
        total  = sum(c['quantite'] for c in cartes)
        status = '✅' if nb_ko == 0 else '⚠️ '
        print(f"  {status} [{deck['tier']}] {deck['nom']:42s} | {len(cartes):2d} uniques | {total:2d} total | ❌ {nb_ko}")
        decks_avec_cartes.append({**deck, 'slug': slug, 'cartes': cartes,
                                   'is_tournament': data.get('is_tournament', '0'),
                                   'views': data.get('views', '0')})
    except Exception as e:
        print(f"  ❌ [{deck['tier']}] {deck['nom']} | {e}")
    time.sleep(0.4)

with open(ROOT / 'data/raw/decklists_raw.json', 'w', encoding='utf-8') as f:
    json.dump(decks_avec_cartes, f, ensure_ascii=False, indent=2)

print(f'\n✅ {len(decks_avec_cartes)} decks récupérés')

🔍 Récupération decklists via API dotgg...

  ✅ [S] Suicune ex Baxcalibur                      | 12 uniques | 20 total | ❌ 0
  ✅ [S] Mega Altaria ex Gourgeist                  | 14 uniques | 20 total | ❌ 0
  ✅ [S] Mega Lucario ex                            | 16 uniques | 20 total | ❌ 0
  ✅ [S] Mega Altaria ex Espeon                     | 13 uniques | 20 total | ❌ 0
  ✅ [S] Mega Lucario ex Greninja                   | 13 uniques | 20 total | ❌ 0
  ✅ [S] Mega Blaziken ex Greninja                  | 14 uniques | 20 total | ❌ 0
  ✅ [S] Mega Manectrix ex Zeraora                  | 12 uniques | 18 total | ❌ 0
  ✅ [S] Zoroark ex Mega Absol ex                   | 14 uniques | 20 total | ❌ 0
  ✅ [S] Hydreigon Mega Absol ex                    | 13 uniques | 20 total | ❌ 0
  ✅ [S] Mega Lucario ex Hitmontop                  | 12 uniques | 20 total | ❌ 0
  ✅ [S] Miraidon ex Magnezone                      | 13 uniques | 20 total | ❌ 0
  ✅ [S] Magnezone Miraidon ex                      | 15 uniques | 

In [4]:
# ============================================================
# [CELL 03] — SCRAPING POKEMONMETA (tier list + winrates + cartes)
# ============================================================

resp_tl   = requests.get('https://www.pokemonmeta.com/tier-list', headers=HEADERS, timeout=15)
resp_tl.raise_for_status()
liens_bruts   = re.findall(r"/tier-list/deck-types/([^\"'<>\s]+)", resp_tl.text)
liens_uniques = list(dict.fromkeys(liens_bruts))
print(f'✅ Tier list pokemonmeta : {len(liens_uniques)} decks\n')

decks_meta = []

for slug in liens_uniques:
    nom = urllib.parse.unquote(slug)
    url = f'https://www.pokemonmeta.com/tier-list/deck-types/{slug}'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, 'html.parser')
        cartes_alt = []
        for img in soup.find_all('img'):
            alt = img.get('alt', '').strip()
            src = img.get('src', '').strip()
            if alt and not src and alt not in ['Pokémon Meta', 'New']:
                key        = normalise(alt)
                matches_c  = cards_index_en.get(key, [])
                type_carte = matches_c[0]['type_carte'] if matches_c else ''
                cartes_alt.append({'nom_en': alt,
                                   'id_carte':   matches_c[0]['id_carte'] if matches_c else 'NON_MATCHÉ',
                                   'nom_fr':     matches_c[0]['nom_fr']   if matches_c else '',
                                   'type_carte': type_carte})
        winrate = meta_id = tour_power = pop_rank = None
        resp_api = requests.get(f'{BASE_META}/deck-types',
                                params={'name': nom, 'aggregate': 'aboveThresh', 'limit': 1},
                                headers=HEADERS, timeout=15)
        if resp_api.status_code == 200 and resp_api.text.strip():
            d_api = resp_api.json()
            if isinstance(d_api, dict) and '_id' in d_api:
                meta_id    = d_api['_id']
                tour_power = d_api.get('tournamentPower')
                pop_rank   = d_api.get('popRank')
                resp_wr = requests.get(f'{BASE_META}/winrates',
                                       params={'decks.deckType': meta_id, 'date[$gt]': '(days-14)'},
                                       headers=HEADERS, timeout=15)
                if resp_wr.status_code == 200 and resp_wr.text.strip():
                    wins = losses = 0
                    for tournoi in resp_wr.json():
                        for deck_entry in tournoi.get('decks', []):
                            if deck_entry.get('deckType', {}).get('_id') == meta_id:
                                for mu in deck_entry.get('matchups', []):
                                    wins   += mu.get('wins', 0)
                                    losses += mu.get('losses', 0)
                    if wins + losses > 0:
                        winrate = round(wins / (wins + losses) * 100, 1)
        nb_ok = sum(1 for c in cartes_alt if c['id_carte'] != 'NON_MATCHÉ')
        print(f'  ✅ {nom:40s} | {len(cartes_alt):2d} cartes | {nb_ok:2d} matchées | winrate={winrate}%')
        decks_meta.append({'nom': nom, 'source': 'pokemonmeta', 'meta_id': meta_id or '',
                           'tournament_power': tour_power, 'pop_rank': pop_rank,
                           'winrate': winrate, 'cartes': cartes_alt})
    except Exception as e:
        print(f'  ❌ {nom} | {e}')
    time.sleep(0.8)

with open(ROOT / 'data/raw/decks_pokemonmeta_raw.json', 'w', encoding='utf-8') as f:
    json.dump(decks_meta, f, ensure_ascii=False, indent=2)

print(f'\n✅ {len(decks_meta)} decks | {sum(1 for d in decks_meta if d["winrate"] is not None)} avec winrate')

✅ Tier list pokemonmeta : 19 decks

  ✅ Miraidon ex Magnezone                    | 19 cartes | 18 matchées | winrate=50.3%
  ✅ Mega Altaria ex Espeon                   | 15 cartes | 14 matchées | winrate=52.6%
  ✅ Suicune ex Baxcalibur                    | 17 cartes | 16 matchées | winrate=53.7%
  ✅ Mega Absol ex Hydreigon                  | 16 cartes | 15 matchées | winrate=52.6%
  ✅ Mega Lucario ex                          | 22 cartes | 22 matchées | winrate=53.9%
  ✅ Mega Lucario ex Greninja                 | 14 cartes | 13 matchées | winrate=53.1%
  ✅ Mega Blaziken ex                         | 18 cartes | 18 matchées | winrate=52.1%
  ✅ Zoroark ex Mega Absol ex                 | 17 cartes | 16 matchées | winrate=52.0%
  ✅ Mega Altaria ex Gourgeist                | 17 cartes | 16 matchées | winrate=52.9%
  ✅ Mega Manectric ex Zeraora                | 18 cartes | 17 matchées | winrate=52.9%
  ✅ Mega Absol ex Darkrai ex                 | 17 cartes | 16 matchées | winrate=52.8%
  ✅ Meg

In [5]:
# ============================================================
# [CELL 04] — SCRAPING GAME8 (tier list + decklists)
# Tier extrait depuis page individuelle — pas de mapping statique
# ============================================================

resp_g8 = requests.get('https://game8.co/games/Pokemon-TCG-Pocket/archives/477754',
                       headers=HEADERS, timeout=15)
resp_g8.raise_for_status()
soup_g8 = BeautifulSoup(resp_g8.text, 'html.parser')
tables  = soup_g8.find_all('table')

MOTS_EXCLUS = ['News', 'Cards', 'Trading', 'Events', 'Ranked', 'Missions',
               'Items', 'Achievements', 'Illustrators', 'Flairs', 'Message',
               'Deck Builder', 'Tools', 'Discord', 'Icon', 'Banner']

decks_game8 = []
for table in tables[1:]:
    for row in table.find_all('tr'):
        for img in row.find_all('img'):
            alt = img.get('alt', '')
            if 'Pokemon TCG Pocket' not in alt:
                continue
            nom = re.sub(r'Pokemon TCG Pocket[\s\-]*', '', alt).strip()
            nom = re.sub(r'\s*(Deck\s*(Image)?|Image)\s*$', '', nom).strip()
            if not nom or len(nom) < 5:
                continue
            if re.search(r'\([A-Z0-9\-]+ \d+\)', nom):
                continue
            if any(m in nom for m in MOTS_EXCLUS):
                continue
            lien = img.find_parent('a')
            if not lien or not lien.get('href'):
                continue
            href = re.sub(r'#.*$', '', lien['href'])
            url  = href if href.startswith('http') else f'https://game8.co{href}'
            if not re.search(r'archives/\d+', url):
                continue
            texte_row = row.get_text()
            m_tier = re.search(r'Rating:\s*([A-S][+\-]?)\s*Tier', texte_row)
            tier   = m_tier.group(1) if m_tier else '?'
            decks_game8.append({'nom': nom, 'tier': tier, 'url': url, 'source': 'game8'})

decks_game8 = list({d['nom']: d for d in decks_game8}.values())
print(f'✅ Tier list game8 : {len(decks_game8)} decks\n')

def parse_decklist_game8(soup):
    cartes = []
    # Tier depuis page individuelle — scalable, pas de mapping statique
    texte_page = soup.get_text()
    m_tier = re.search(r'Rating:\s*([A-S][+\-]?)\s*Tier', texte_page)
    tier   = m_tier.group(1) if m_tier else '?'
    for table in soup.find_all('table'):
        texte = table.get_text(strip=True)
        if '×' not in texte or 'Expansion' in texte or 'Tier' in texte:
            continue
        matches = re.findall(r"([A-Za-zéèàêëïîôùûç\'\-\. ]+?)×(\d)", texte)
        for nom_en, qty in matches:
            nom_en = nom_en.strip()
            if len(nom_en) < 2:
                continue
            key        = normalise(nom_en)
            matches_c  = cards_index_en.get(key, [])
            type_carte = matches_c[0]['type_carte'] if matches_c else ''
            cartes.append({'nom_en': nom_en, 'quantite': int(qty),
                           'id_carte':   matches_c[0]['id_carte'] if matches_c else 'NON_MATCHÉ',
                           'nom_fr':     matches_c[0]['nom_fr']   if matches_c else '',
                           'type_carte': type_carte,
                           'code_set':   str(matches_c[0]['code_set']) if matches_c else '',
                           'numero':     str(matches_c[0]['numero'])   if matches_c else ''})
        if cartes:
            break
    return cartes, tier

print('🔍 Scraping decklists game8...\n')
decks_game8_complets = []

for deck in decks_game8:
    try:
        resp          = requests.get(deck['url'], headers=HEADERS, timeout=15)
        soup          = BeautifulSoup(resp.text, 'html.parser')
        cartes, tier  = parse_decklist_game8(soup)
        tier_final    = tier if tier != '?' else deck.get('tier', '?')
        nb_ok  = sum(1 for c in cartes if c['id_carte'] != 'NON_MATCHÉ')
        total  = sum(c['quantite'] for c in cartes)
        status = '✅' if len(cartes) > 0 else '⚠️ '
        print(f"  {status} [{tier_final:2s}] {deck['nom']:42s} | {len(cartes):2d} uniques | {total:2d} total | ❌ {len(cartes)-nb_ok}")
        decks_game8_complets.append({**deck, 'tier': tier_final, 'cartes': cartes})
    except Exception as e:
        print(f"  ❌ {deck['nom']} | {e}")
        decks_game8_complets.append({**deck, 'cartes': []})
    time.sleep(0.5)

with open(ROOT / 'data/raw/decks_game8_raw.json', 'w', encoding='utf-8') as f:
    json.dump(decks_game8_complets, f, ensure_ascii=False, indent=2)

print(f'\n✅ {len(decks_game8_complets)} decks | {sum(1 for d in decks_game8_complets if d["cartes"])} avec cartes')

✅ Tier list game8 : 10 decks

🔍 Scraping decklists game8...

  ✅ [? ] Mega Lucario ex and Hitmontop              | 12 uniques | 20 total | ❌ 0
  ✅ [? ] Miraidon ex and CB Magnezone               | 13 uniques | 20 total | ❌ 0
  ✅ [? ] Mega Altaria ex and PD Espeon              | 13 uniques | 19 total | ❌ 0
  ✅ [? ] Magnezone ex and Pom-Pom Oricorio          | 14 uniques | 20 total | ❌ 0
  ✅ [? ] Iron Valiant and Mega Altaria ex           | 11 uniques | 20 total | ❌ 0
  ✅ [? ] Mega Lucario ex and Darkrai                | 12 uniques | 20 total | ❌ 0
  ✅ [? ] Mega Sceptile ex and Teal Mask Ogerpon ex  | 14 uniques | 20 total | ❌ 0
  ✅ [? ] Darkrai and Mega Altaria ex                | 13 uniques | 19 total | ❌ 0
  ✅ [? ] Chien-Pao ex and Baxcalibur                | 13 uniques | 20 total | ❌ 0
  ✅ [? ] Mega Altaria ex and Gourgeist              | 13 uniques | 20 total | ❌ 0

✅ 10 decks | 10 avec cartes


In [6]:
# ============================================================
# [CELL 05] — ANALYSE STAPLES UNIVERSELS
# FIX : dédoublonnage sur id_carte + nunique sur id_deck
# ============================================================

rows = []
for d in decks_avec_cartes:
    for c in d['cartes']:
        rows.append({'source': 'ptcgpocket', 'id_deck': d['slug'], 'tier': d['tier'],
                     'id_carte': c['id_carte'], 'nom_fr': c['nom_fr'],
                     'type_carte': c['type_carte'], 'quantite': c['quantite']})
for d in decks_meta:
    for c in d['cartes']:
        if c['id_carte'] != 'NON_MATCHÉ':
            rows.append({'source': 'pokemonmeta', 'id_deck': urllib.parse.quote(d['nom']), 'tier': '?',
                         'id_carte': c['id_carte'], 'nom_fr': c['nom_fr'],
                         'type_carte': c.get('type_carte', ''), 'quantite': 1})
for d in decks_game8_complets:
    for c in d['cartes']:
        if c['id_carte'] != 'NON_MATCHÉ':
            rows.append({'source': 'game8', 'id_deck': urllib.parse.quote(d['nom']), 'tier': d['tier'],
                         'id_carte': c['id_carte'], 'nom_fr': c['nom_fr'],
                         'type_carte': c.get('type_carte', ''), 'quantite': c['quantite']})

df_all_cards   = pd.DataFrame(rows)
nb_decks_total = len(decks_avec_cartes) + len(decks_meta) + len(decks_game8_complets)

type_carte_map = (df_all_cards[df_all_cards['type_carte'] != '']
                  .drop_duplicates('id_carte')
                  .set_index('id_carte')['type_carte'])

staples = (df_all_cards.groupby(['id_carte', 'nom_fr'])
           .agg(nb_decks=('id_deck', 'nunique'), quantite_moy=('quantite', 'mean'))
           .reset_index()
           .sort_values('nb_decks', ascending=False))
staples['type_carte']   = staples['id_carte'].map(type_carte_map).fillna('')
staples['pct_decks']    = (staples['nb_decks'] / nb_decks_total * 100).round(1)
staples['quantite_moy'] = staples['quantite_moy'].round(2)
staples = staples[['id_carte', 'nom_fr', 'type_carte', 'nb_decks', 'quantite_moy', 'pct_decks']]

print(f'📊 STAPLES UNIVERSELS — Top 30 (sur {nb_decks_total} decks)\n')
print(f"  {'Carte':28s} {'Type':25s} {'Decks':>6} {'%':>6} {'Qté moy':>8}")
print('  ' + '-' * 78)
for _, r in staples.head(30).iterrows():
    print(f"  {r['nom_fr']:28s} {r['type_carte']:25s} {r['nb_decks']:>6} {r['pct_decks']:>5}% {r['quantite_moy']:>8.2f}")
print(f'\n✅ {len(staples)} staples uniques')

📊 STAPLES UNIVERSELS — Top 30 (sur 55 decks)

  Carte                        Type                       Decks      %  Qté moy
  ------------------------------------------------------------------------------
  Copieuse                     dresseur-supporter            55 100.0%     1.38
  Recherches Professorales     dresseur-supporter            55 100.0%     1.62
  Hélio                        dresseur-supporter            52  94.5%     1.02
  Poké Ball                    dresseur-objet                48  87.3%     1.62
  Morgane                      dresseur-supporter            34  61.8%     1.00
  Dame du Centre Pokémon       dresseur-supporter            20  36.4%     1.00
  Cape Géante                  dresseur-outil pokémon        18  32.7%     1.11
  Super Bonbon                 dresseur-objet                18  32.7%     1.50
  Atalante                     dresseur-supporter            16  29.1%     1.00
  Casque Brut                  dresseur-outil pokémon        16  29.1%   

In [7]:
# ============================================================
# [CELL 06] — MATRICE MATCHUPS
# Source : pokemonmeta API /winrates (14 jours)
# ============================================================

print('🔍 Récupération matchups pokemonmeta...\n')
rows_matchups = []

for d in decks_meta:
    meta_id  = d.get('meta_id', '')
    nom_deck = d['nom']
    if not meta_id:
        continue
    try:
        resp = requests.get(f'{BASE_META}/winrates',
                            params={'decks.deckType': meta_id, 'date[$gt]': '(days-14)'},
                            headers=HEADERS, timeout=15)
        if resp.status_code != 200 or not resp.text.strip():
            print(f'  ⚠️  {nom_deck} — réponse vide')
            continue
        matchups_agg = {}
        for tournoi in resp.json():
            for deck_entry in tournoi.get('decks', []):
                if deck_entry.get('deckType', {}).get('_id') != meta_id:
                    continue
                for mu in deck_entry.get('matchups', []):
                    adv_id  = mu.get('deckType', {}).get('_id', '')
                    adv_nom = mu.get('deckType', {}).get('name', '')
                    if not adv_id:
                        continue
                    if adv_id not in matchups_agg:
                        matchups_agg[adv_id] = {'nom_adv': adv_nom, 'wins': 0, 'losses': 0}
                    matchups_agg[adv_id]['wins']   += mu.get('wins', 0)
                    matchups_agg[adv_id]['losses'] += mu.get('losses', 0)
        nb_mu = 0
        for adv_id, data in matchups_agg.items():
            total = data['wins'] + data['losses']
            if total == 0:
                continue
            rows_matchups.append({'meta_id_deck': meta_id, 'nom_deck_en': nom_deck,
                                  'meta_id_adv': adv_id, 'nom_adv_en': data['nom_adv'],
                                  'wins': data['wins'], 'losses': data['losses'],
                                  'total_games': total,
                                  'winrate': round(data['wins'] / total * 100, 1)})
            nb_mu += 1
        print(f'  ✅ {nom_deck:42s} | {nb_mu} adversaires')
    except Exception as e:
        print(f'  ❌ {nom_deck} | {e}')
    time.sleep(0.5)

df_matchups = pd.DataFrame(rows_matchups)
print(f'\n✅ {len(df_matchups)} matchups | {df_matchups["nom_deck_en"].nunique()} decks | {df_matchups["nom_adv_en"].nunique()} adversaires')

SEUIL_MIN = 5
print(f'\n📊 Top favorables (winrate > 55%, min {SEUIL_MIN} parties) :')
top = df_matchups[(df_matchups['winrate'] > 55) & (df_matchups['total_games'] >= SEUIL_MIN)].sort_values('winrate', ascending=False)
print(top[['nom_deck_en','nom_adv_en','winrate','total_games']].head(15).to_string(index=False))
print(f'\n📊 Top défavorables (winrate < 45%, min {SEUIL_MIN} parties) :')
bot = df_matchups[(df_matchups['winrate'] < 45) & (df_matchups['total_games'] >= SEUIL_MIN)].sort_values('winrate')
print(bot[['nom_deck_en','nom_adv_en','winrate','total_games']].head(15).to_string(index=False))

🔍 Récupération matchups pokemonmeta...

  ✅ Miraidon ex Magnezone                      | 154 adversaires
  ✅ Mega Altaria ex Espeon                     | 137 adversaires
  ✅ Suicune ex Baxcalibur                      | 141 adversaires
  ✅ Mega Absol ex Hydreigon                    | 117 adversaires
  ✅ Mega Lucario ex                            | 123 adversaires
  ✅ Mega Lucario ex Greninja                   | 107 adversaires
  ✅ Mega Blaziken ex                           | 141 adversaires
  ✅ Zoroark ex Mega Absol ex                   | 115 adversaires
  ✅ Mega Altaria ex Gourgeist                  | 121 adversaires
  ✅ Mega Manectric ex Zeraora                  | 106 adversaires
  ✅ Mega Absol ex Darkrai ex                   | 101 adversaires
  ✅ Mega Charizard Y ex                        | 71 adversaires
  ✅ Mega Altaria ex Greninja                   | 66 adversaires
  ✅ Meowscarada ex                             | 89 adversaires
  ✅ Koraidon ex Great Tusk                     | 92 a

In [8]:
# ============================================================
# [CELL 06b] — FIX NOMS FR ADVERSAIRES MATCHUPS
# Résolution via nom_fr_depuis_nom_en — sans appel API, scalable
# ============================================================

# Index meta_id → nom_fr depuis nos decks pokemonmeta
meta_id_to_nom_fr = {}
for d in decks_meta:
    mid = d.get('meta_id', '')
    if not mid:
        continue
    id_deck = re.sub(r'[^a-z0-9]+', '-', d['nom'].lower()).strip('-')
    cartes  = d.get('cartes', [])
    if not cartes:
        continue
    df_temp = pd.DataFrame([{'id_deck': id_deck, **c} for c in cartes])
    nom_fr  = construire_nom_fr_deck(id_deck, df_temp)
    meta_id_to_nom_fr[mid] = nom_fr if nom_fr else nom_fr_depuis_nom_en(d['nom'])

# Adversaires non mappés → résolution via nom EN
adv_ids_non_mappes = set(df_matchups['meta_id_adv']) - set(meta_id_to_nom_fr.keys())
print(f'🔍 Résolution {len(adv_ids_non_mappes)} adversaires non mappés...\n')

resolus = 0
for adv_id in adv_ids_non_mappes:
    nom_en = df_matchups[df_matchups['meta_id_adv'] == adv_id]['nom_adv_en'].iloc[0]
    nom_fr = nom_fr_depuis_nom_en(nom_en)
    meta_id_to_nom_fr[adv_id] = nom_fr
    if nom_fr != nom_en:
        resolus += 1

# Mise à jour df_matchups
df_matchups['nom_fr_deck'] = df_matchups['meta_id_deck'].map(meta_id_to_nom_fr).fillna(df_matchups['nom_deck_en'])
df_matchups['nom_fr_adv']  = df_matchups['meta_id_adv'].map(meta_id_to_nom_fr).fillna(df_matchups['nom_adv_en'])

encore_en = df_matchups[df_matchups['nom_fr_adv'] == df_matchups['nom_adv_en']]['meta_id_adv'].nunique()
print(f'✅ Adversaires résolus : {resolus} | Encore en EN : {encore_en}')
print(f'\n📊 Aperçu matchups FR :')
print(df_matchups[['nom_fr_deck','nom_fr_adv','winrate','total_games']].head(10).to_string(index=False))

🔍 Résolution 173 adversaires non mappés...

✅ Adversaires résolus : 161 | Encore en EN : 12

📊 Aperçu matchups FR :
               nom_fr_deck                                   nom_fr_adv  winrate  total_games
Miraidon-ex / Magnézone-ex                  Méga-Cizayox-ex / Vrombotor     30.1          123
Miraidon-ex / Magnézone-ex                    Méga-Altaria-ex / Darkrai     50.6          411
Miraidon-ex / Magnézone-ex Méga-Jungko-ex / Ogerpon Masque Turquoise-ex     58.6          133
Miraidon-ex / Magnézone-ex               Entei-ex / Méga-Dracaufeu Y-ex     48.1           81
Miraidon-ex / Magnézone-ex                    Koraidon-ex / Fort-Ivoire     41.2          102
Miraidon-ex / Magnézone-ex                                    Plumeline     70.0           10
Miraidon-ex / Magnézone-ex  Miascarade-ex / Ogerpon Masque Turquoise-ex     66.3          104
Miraidon-ex / Magnézone-ex                            Méga-Braségali-ex     40.7          423
Miraidon-ex / Magnézone-ex            

In [9]:
# ============================================================
# [CELL 07] — RÉSULTATS TOURNOIS
# Source : pokemonmeta API /top-decks
# ============================================================

print('🔍 Récupération résultats tournois...\n')
rows_tournois = []
page = 1
limit = 100
total_fetched = 0

while True:
    resp = requests.get(f'{BASE_META}/top-decks',
                        params={'limit': limit, 'skip': (page - 1) * limit,
                                'sort': '-date', 'aggregate': 'sortByPlacement', 'fields': '-side'},
                        headers=HEADERS, timeout=15)
    if resp.status_code != 200 or not resp.text.strip():
        break
    data  = resp.json()
    items = data if isinstance(data, list) else [data]
    if not items:
        break
    for item in items:
        deck_type = item.get('deckType', {})
        nom_en    = deck_type.get('name', '')
        nom_fr    = nom_fr_depuis_nom_en(nom_en)
        cartes = []
        for c in item.get('main', []):
            card    = c.get('card', {})
            nom_c   = card.get('name', '')
            key     = normalise(nom_c)
            match_c = cards_index_en.get(key, [])
            cartes.append({'nom_en': nom_c,
                           'nom_fr':   match_c[0]['nom_fr']   if match_c else nom_c,
                           'id_carte': match_c[0]['id_carte'] if match_c else '',
                           'quantite': c.get('amount', 1)})
        rows_tournois.append({'id_tournoi':   item.get('_id', ''),
                              'date':         item.get('created', '')[:10],
                              'auteur':       item.get('author', ''),
                              'nom_deck_en':  nom_en,
                              'nom_deck_fr':  nom_fr,
                              'placement':    item.get('tournamentPlacement', ''),
                              'num_tournoi':  item.get('tournamentNumber', ''),
                              'type_tournoi': item.get('tournamentType', {}).get('name', ''),
                              'url':          item.get('url', ''),
                              'nb_cartes':    len(cartes),
                              'cartes_json':  json.dumps(cartes, ensure_ascii=False)})
    total_fetched += len(items)
    print(f'  Page {page} : {len(items)} résultats (total : {total_fetched})')
    if len(items) < limit:
        break
    page += 1
    time.sleep(0.5)

df_tournois = pd.DataFrame(rows_tournois)
print(f'\n✅ {len(df_tournois)} résultats | {df_tournois["num_tournoi"].nunique()} tournois | période : {df_tournois["date"].min()} → {df_tournois["date"].max()}')
print(f'\n📊 Top decks gagnants (1st Place) :')
wins = df_tournois[df_tournois['placement'] == '1st Place']
print(wins.groupby('nom_deck_fr')['id_tournoi'].count().sort_values(ascending=False).head(10).to_string())

🔍 Récupération résultats tournois...

  Page 1 : 100 résultats (total : 100)
  Page 2 : 100 résultats (total : 200)
  Page 3 : 100 résultats (total : 300)
  Page 4 : 100 résultats (total : 400)
  Page 5 : 100 résultats (total : 500)
  Page 6 : 100 résultats (total : 600)
  Page 7 : 100 résultats (total : 700)
  Page 8 : 100 résultats (total : 800)
  Page 9 : 26 résultats (total : 826)

✅ 826 résultats | 28 tournois | période : 2026-06-02 → 2026-06-16

📊 Top decks gagnants (1st Place) :
nom_deck_fr
Miraidon-ex / Magnézone           9
Méga-Braségali-ex                 7
Méga-Altaria-ex / Mentali         7
Suicune-ex / Glaivodo             6
Méga-Lucario-ex                   5
Méga-Absol-ex / Trioxhydre        4
Méga-Élecsprint-ex / Zeraora      4
Méga-Lucario-ex / Amphinobi       3
Zoroark-ex / Méga-Absol-ex        3
Méga-Altaria-ex / Banshitrouye    3


In [10]:
# ============================================================
# [CELL 08] — GÉNÉRATION NOMS FR + EXPORT CSV
# FIX : noms FR decks, id_deck slug propre, pokemon_cles ;
# ============================================================

TODAY         = date.today().isoformat()
SAISON_ACTIVE = 'B2b'

# --- SOURCE 1 : ptcgpocket ---
rows_decks, rows_cards = [], []
for d in decks_avec_cartes:
    cartes = d['cartes']
    nb_ex  = sum(1 for c in cartes if 'ex' in c['nom_fr'].lower() and c['type_carte'] == 'pokemon')
    pokes  = list(dict.fromkeys(c['nom_fr'] for c in cartes if c['type_carte'] == 'pokemon'))
    rows_decks.append({'id_deck': d['slug'], 'nom': d['nom'], 'source': 'ptcgpocket',
                       'tier': d['tier'], 'tendance': d['tendance'], 'saison_active': SAISON_ACTIVE,
                       'is_tournament': d.get('is_tournament', '0'), 'views': d.get('views', '0'),
                       'nb_cartes_uniques': len(cartes), 'nb_ex': nb_ex,
                       'pokemon_cles': ' ; '.join(pokes), 'date_scraping': TODAY})
    for c in cartes:
        rows_cards.append({'id_deck': d['slug'], 'nom_deck': d['nom'], 'source': 'ptcgpocket',
                           'id_carte': c['id_carte'], 'nom_fr': c['nom_fr'], 'nom_en': c['nom_en'],
                           'type_carte': c['type_carte'], 'code_set': c['code_set'],
                           'numero': c['numero'], 'quantite': c['quantite']})
df_decks_ptcg = pd.DataFrame(rows_decks)
df_cards_ptcg = pd.DataFrame(rows_cards)

# --- SOURCE 2 : pokemonmeta ---
rows_decks, rows_cards = [], []
for d in decks_meta:
    id_deck = re.sub(r'[^a-z0-9]+', '-', d['nom'].lower()).strip('-')
    rows_decks.append({'id_deck': id_deck, 'nom': d['nom'], 'source': 'pokemonmeta',
                       'saison_active': SAISON_ACTIVE, 'meta_id': d.get('meta_id', ''),
                       'winrate': d.get('winrate'), 'tournament_power': d.get('tournament_power'),
                       'pop_rank': d.get('pop_rank'),
                       'nb_cartes': len(d['cartes']),
                       'nb_matchees': sum(1 for c in d['cartes'] if c['id_carte'] != 'NON_MATCHÉ'),
                       'date_scraping': TODAY})
    for c in d['cartes']:
        rows_cards.append({'id_deck': id_deck, 'nom_deck': d['nom'], 'source': 'pokemonmeta',
                           'nom_en': c['nom_en'], 'nom_fr': c['nom_fr'],
                           'id_carte': c['id_carte'], 'type_carte': c.get('type_carte', '')})
df_meta_decks = pd.DataFrame(rows_decks)
df_cards_meta = pd.DataFrame(rows_cards)

# --- SOURCE 3 : game8 ---
rows_decks, rows_cards = [], []
for d in decks_game8_complets:
    id_deck = re.sub(r'[^a-z0-9]+', '-', d['nom'].lower()).strip('-')
    rows_decks.append({'id_deck': id_deck, 'nom': d['nom'], 'source': 'game8',
                       'saison_active': SAISON_ACTIVE, 'tier': d['tier'],
                       'nb_cartes_uniques': len(d['cartes']),
                       'nb_matchees': sum(1 for c in d['cartes'] if c['id_carte'] != 'NON_MATCHÉ'),
                       'date_scraping': TODAY})
    for c in d['cartes']:
        rows_cards.append({'id_deck': id_deck, 'nom_deck': d['nom'], 'source': 'game8',
                           'nom_en': c['nom_en'], 'nom_fr': c['nom_fr'],
                           'id_carte': c['id_carte'], 'type_carte': c.get('type_carte', ''),
                           'code_set': c['code_set'], 'numero': c['numero'], 'quantite': c['quantite']})
df_decks_g8 = pd.DataFrame(rows_decks)
df_cards_g8 = pd.DataFrame(rows_cards)

# --- GÉNÉRATION NOM FR (toutes sources) ---
df_decks_ptcg['nom_fr'] = df_decks_ptcg['id_deck'].apply(lambda x: construire_nom_fr_deck(x, df_cards_ptcg))
df_meta_decks['nom_fr'] = df_meta_decks['id_deck'].apply(lambda x: construire_nom_fr_deck(x, df_cards_meta))
df_decks_g8['nom_fr']   = df_decks_g8['id_deck'].apply(lambda x: construire_nom_fr_deck(x, df_cards_g8))

# Ajout nom_fr_deck dans deck_cards
for df_d, df_c in [(df_decks_ptcg, df_cards_ptcg), (df_meta_decks, df_cards_meta), (df_decks_g8, df_cards_g8)]:
    map_fr = df_d.set_index('id_deck')['nom_fr'].to_dict()
    df_c['nom_fr_deck'] = df_c['id_deck'].map(map_fr)

# --- EXPORT CSV ---
df_decks_ptcg.to_csv(OUT_DECKS_PTCG, index=False, encoding='utf-8-sig')
df_cards_ptcg.to_csv(OUT_CARDS_PTCG, index=False, encoding='utf-8-sig')
df_meta_decks.to_csv(OUT_DECKS_META, index=False, encoding='utf-8-sig')
df_cards_meta.to_csv(OUT_CARDS_META, index=False, encoding='utf-8-sig')
df_decks_g8.to_csv(ROOT / 'data/processed/decks_game8.csv',      index=False, encoding='utf-8-sig')
df_cards_g8.to_csv(ROOT / 'data/processed/deck_cards_game8.csv', index=False, encoding='utf-8-sig')
staples.to_csv(ROOT / 'data/processed/staples.csv',              index=False, encoding='utf-8-sig')
df_matchups.to_csv(ROOT / 'data/processed/matchups.csv',         index=False, encoding='utf-8-sig')
df_tournois.to_csv(ROOT / 'data/processed/tournois.csv',         index=False, encoding='utf-8-sig')

print('✅ EXPORT CSV COMPLET\n')
print(f'📄 decks_ptcgpocket      : {len(df_decks_ptcg)} decks | {len(df_cards_ptcg)} lignes')
print(f'📄 decks_pokemonmeta     : {len(df_meta_decks)} decks | {len(df_cards_meta)} lignes')
print(f'📄 decks_game8           : {len(df_decks_g8)} decks | {len(df_cards_g8)} lignes')
print(f'📄 staples               : {len(staples)} cartes')
print(f'📄 matchups              : {len(df_matchups)} lignes')
print(f'📄 tournois              : {len(df_tournois)} lignes')
print('\n📋 Noms FR — ptcgpocket :')
for _, r in df_decks_ptcg[['nom', 'nom_fr', 'tier']].iterrows():
    print(f"  [{r['tier']}] {r['nom']:42s} → {r['nom_fr']}")

✅ EXPORT CSV COMPLET

📄 decks_ptcgpocket      : 26 decks | 348 lignes
📄 decks_pokemonmeta     : 19 decks | 321 lignes
📄 decks_game8           : 10 decks | 128 lignes
📄 staples               : 149 cartes
📄 matchups              : 2037 lignes
📄 tournois              : 826 lignes

📋 Noms FR — ptcgpocket :
  [S] Suicune ex Baxcalibur                      → Suicune-ex / Baojian-ex
  [S] Mega Altaria ex Gourgeist                  → Méga-Altaria-ex / Pitrouille
  [S] Mega Lucario ex                            → Méga-Lucario-ex / Kicklee
  [S] Mega Altaria ex Espeon                     → Méga-Altaria-ex / Toudoudou
  [S] Mega Lucario ex Greninja                   → Méga-Lucario-ex / Grenousse
  [S] Mega Blaziken ex Greninja                  → Méga-Braségali-ex / Poussifeu
  [S] Mega Manectrix ex Zeraora                  → Méga-Élecsprint-ex / Zeraora
  [S] Zoroark ex Mega Absol ex                   → Zoroark-ex / Darkrai-ex
  [S] Hydreigon Mega Absol ex                    → Méga-Absol-ex / Sol

In [16]:
# ============================================================
# [CELL 09] — EXPORT GOOGLE SHEETS
# ============================================================

import gspread
from google.oauth2.service_account import Credentials

SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]
creds = Credentials.from_service_account_file(
    str(ROOT / 'config' / 'service_account.json'), scopes=SCOPES
)
gc = gspread.authorize(creds)

with open(ROOT / 'config' / 'sheets_ids.json') as f:
    SHEETS_IDS = json.load(f)

print('📤 Export Google Sheets...\n')
export_to_sheets(df_decks_ptcg, 'decks_ptcgpocket',       gc, SHEETS_IDS)
export_to_sheets(df_cards_ptcg, 'deck_cards_ptcgpocket',  gc, SHEETS_IDS)
export_to_sheets(df_meta_decks, 'decks_pokemonmeta',      gc, SHEETS_IDS)
export_to_sheets(df_cards_meta, 'deck_cards_pokemonmeta', gc, SHEETS_IDS)
export_to_sheets(df_decks_g8,   'decks_game8',            gc, SHEETS_IDS)
export_to_sheets(df_cards_g8,   'deck_cards_game8',       gc, SHEETS_IDS)
export_to_sheets(staples,       'staples',                gc, SHEETS_IDS)
export_to_sheets(df_matchups,   'matchups',               gc, SHEETS_IDS)
export_to_sheets(df_tournois,   'tournois',               gc, SHEETS_IDS)
print('\n✅ Sync Sheets terminé')

📤 Export Google Sheets...



C:\Users\JPVT\AppData\Local\Temp\ipykernel_10348\3532039243.py:140: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_clean = df_clean.infer_objects(copy=False).fillna('').astype(str)


  ✅ decks_ptcgpocket          → 26 lignes


C:\Users\JPVT\AppData\Local\Temp\ipykernel_10348\3532039243.py:140: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_clean = df_clean.infer_objects(copy=False).fillna('').astype(str)


  ✅ deck_cards_ptcgpocket     → 348 lignes


...

  ✅ staples                   → 149 lignes


C:\Users\JPVT\AppData\Local\Temp\ipykernel_10348\3532039243.py:140: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_clean = df_clean.infer_objects(copy=False).fillna('').astype(str)


  ✅ matchups                  → 2037 lignes


C:\Users\JPVT\AppData\Local\Temp\ipykernel_10348\3532039243.py:140: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df_clean = df_clean.infer_objects(copy=False).fillna('').astype(str)


  ✅ tournois                  → 826 lignes

✅ Sync Sheets terminé
